In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/,but they won't be saved outside of the current session

In [3]:
import torch
print(torch.__version__)          # هتشوف مثلاً 2.1.0+cu118
print(torch.version.cuda)         # هتشوف مثلاً 11.8

2.10.0+cu128
12.8


In [ ]:
# # ✅ البديل: install من source مباشرة
# !pip install torch-geometric -q
# !pip install torch-scatter -q
# !pip install torch-sparse -q

# print("✅ Done")

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done


In [ ]:
# import torch

# # تحديد الـ Version بتاع Torch و CUDA عشان نحمل الملفات المتوافقة فوراً
# TORCH = torch.__version__.split('+')[0]
# CUDA = "cu" + torch.version.cuda.replace('.', '')

# print(f"Detected: Torch {TORCH} | CUDA {CUDA}")

# # تثبيت المكتبات من الـ binaries الجاهزة (أسرع بـ 10 مرات)
# !pip install torch-scatter -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html -q
# !pip install torch-sparse -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html -q
# !pip install torch-geometric -q

# print("✅ GNN Environment Ready!")

In [4]:
# # ✅ Step 1: install torch-geometric base
# !pip install torch-geometric -q

# # ✅ Step 2: install scatter & sparse with cu121 (أقرب version متاحة لـ cu128)
# !pip install torch-scatter torch-sparse \
#     -f https://data.pyg.org/whl/torch-2.1.0+cu121.html -q

# print("✅ Done")

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
^C
ERROR: Operation cancelled by user
✅ Done


In [4]:
import torch
import os

# ── Get torch version for correct wheels──
torch_version  = torch.__version__.split("+")[0]   # e.g. "2.1.0"
cuda_version   = "cu118"                            # Kaggle default CUDA

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")

# ── Install torch-geometric + dependencies ──
os.system(f"pip install torch-geometric -q")
os.system(
    f"pip install torch-scatter torch-sparse "
    f"-f https://data.pyg.org/whl/torch-{torch_version}+{cuda_version}.html -q"
)

# ── Verify ──
try:
    from torch_geometric.nn import GATv2Conv
    from torch_geometric.data import Data, Batch
    print("\n[✓] torch_geometric installed successfully")
except ImportError as e:
    print(f"\n[✗] Still not found: {e}")
    print("Try restarting the kernel then run again")

PyTorch : 2.10.0+cu128
CUDA    : 12.8

[✓] torch_geometric installed successfully


In [5]:
"""
=============================================================
  GNN Fusion + MaxViT-T  —  Multimodal Skin Cancer Classifier
  ✅ Updated to use pre-split train/val/test data
=============================================================
"""

# ─────────────────────────────────────────────
# 0.  Imports
# ─────────────────────────────────────────────
import os, re, warnings
import numpy as np
import pandas as pd
import cv2
warnings.filterwarnings("ignore")

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

import albumentations as A
from albumentations.pytorch import ToTensorV2

import timm
from tqdm import tqdm

from torch_geometric.nn import GATv2Conv
from torch_geometric.data import Data, Batch

# ─────────────────────────────────────────────
# 1.  CFG
# ─────────────────────────────────────────────
class CFG:
    IMG_SIZE      = 224
    MEAN          = [0.485, 0.456, 0.406]
    STD           = [0.229, 0.224, 0.225]
    D             = 256
    GAT_HEADS     = 4
    META_TOKENS   = 4
    DROPOUT       = 0.50
    FREEZE_BLOCKS = 2
    EPOCHS        = 20
    BATCH_SIZE    = 16
    GRAD_ACCUM    = 2
    NUM_WORKERS   = 2
    PATIENCE      = 10
    SEED          = 42
    LABEL_SMOOTH  = 0.15
    FOCAL_ALPHA   = 0.25
    FOCAL_GAMMA   = 2.0
    LR            = 1e-5
    HEAD_LR       = 2e-4
    WEIGHT_DECAY  = 1e-2
    TTA_STEPS     = 5

    # ✅ Paths
    BASE_DIR      = "/kaggle/input/datasets/malakaboelmagd/splitted-data-set"
    TARGET_COL    = "class"

# ─────────────────────────────────────────────
# 2.  Device & Seed
# ─────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.manual_seed(CFG.SEED)
np.random.seed(CFG.SEED)

# ─────────────────────────────────────────────
# 3.  Load Pre-Split Data ✅
# ─────────────────────────────────────────────
def load_split(split_name):
    # ✅ handles double-nested: base/train/train/train.csv
    split_folder = os.path.join(CFG.BASE_DIR, split_name, split_name)
    csv_path     = os.path.join(split_folder, f"{split_name}.csv")
    images_dir   = os.path.join(split_folder, "images")

    df = pd.read_csv(csv_path)
    df["images_dir"] = images_dir
    return df

train_df = load_split("train")
val_df   = load_split("val")
test_df  = load_split("test")

# Build unified label mapping from training set
classes      = sorted(train_df[CFG.TARGET_COL].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
NUM_CLASSES  = len(classes)
classes_str  = [str(c) for c in classes]

for df_ in [train_df, val_df, test_df]:
    df_["label"] = df_[CFG.TARGET_COL].map(class_to_idx)

print(f"Classes     : {classes_str}")
print(f"Num classes : {NUM_CLASSES}")
print(f"Train size  : {len(train_df)}")
print(f"Val size    : {len(val_df)}")
print(f"Test size   : {len(test_df)}")
print(f"\nTrain distribution:\n{train_df['label'].value_counts()}")

# ─────────────────────────────────────────────
# 4.  Image Name Fix
# ─────────────────────────────────────────────
def normalize_name(fname):
    x    = str(fname).replace(".jpg", "")
    nums = re.findall(r'(\d{7})', x)
    if nums:
        return f"ISIC_{nums[-1]}.jpg"
    return None

for df_ in [train_df, val_df, test_df]:
    if "image_fixed" not in df_.columns:
        df_["image_fixed"] = df_["image"].apply(normalize_name)
    df_.dropna(subset=["image_fixed"], inplace=True)
    df_.reset_index(drop=True, inplace=True)

# ─────────────────────────────────────────────
# 5.  META_COLS
# ─────────────────────────────────────────────
DROP_COLS = ["image", "isic_id", "patient_id", "year",
             "class", "image_fixed", "images_dir"]
META_COLS = [c for c in train_df.columns
             if c not in DROP_COLS + ["label"]]
META_DIM  = len(META_COLS)
print(f"\nMeta columns ({META_DIM}): {META_COLS}")

# ─────────────────────────────────────────────
# 6.  Augmentation
# ─────────────────────────────────────────────
def get_train_transform(img_size=CFG.IMG_SIZE):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Rotate(limit=15, p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05,
                           rotate_limit=10, p=0.4),
        A.RandomBrightnessContrast(brightness_limit=0.1,
                                   contrast_limit=0.1, p=0.4),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.2),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.Sharpen(alpha=(0.1, 0.2), lightness=(0.9, 1.0), p=1.0),
        ], p=0.1),
        A.Normalize(mean=CFG.MEAN, std=CFG.STD),
        ToTensorV2()
    ])

def get_val_transform(img_size=CFG.IMG_SIZE):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=CFG.MEAN, std=CFG.STD),
        ToTensorV2()
    ])

# ─────────────────────────────────────────────
# 7.  Dataset ✅ uses per-row images_dir
# ─────────────────────────────────────────────
class GNNDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(row["images_dir"], row["image_fixed"])
        img      = cv2.imread(img_path)

        if img is None:
            return self.__getitem__((idx + 1) % len(self.df))

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (CFG.IMG_SIZE, CFG.IMG_SIZE))

        if self.transform:
            img = self.transform(image=img)["image"]

        meta  = torch.tensor(row[META_COLS].values.astype(np.float32))
        label = torch.tensor(row["label"], dtype=torch.long)
        return img, meta, label

# ─────────────────────────────────────────────
# 8.  Graph Construction
# ─────────────────────────────────────────────
def build_graph(img_feat, meta_feat):
    R = img_feat.size(0)
    M = meta_feat.size(0)
    x = torch.cat([img_feat, meta_feat], dim=0)

    edges = []
    for i in range(R):
        for j in range(R):
            edges.append([i, j])
    for m in range(R, R + M):
        for i in range(R):
            edges.append([m, i])
            edges.append([i, m])

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return Data(x=x, edge_index=edge_index)

# ─────────────────────────────────────────────
# 9.  Focal Loss + Label Smoothing
# ─────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, logits, targets):
        num_classes = logits.size(1)
        with torch.no_grad():
            smooth = torch.full_like(
                logits, CFG.LABEL_SMOOTH / (num_classes - 1))
            smooth.scatter_(1, targets.unsqueeze(1), 1.0 - CFG.LABEL_SMOOTH)
        log_prob = F.log_softmax(logits, dim=1)
        ce       = -(smooth * log_prob).sum(dim=1)
        pt       = torch.exp(-ce)
        loss     = CFG.FOCAL_ALPHA * (1 - pt) ** CFG.FOCAL_GAMMA * ce
        return loss.mean()

# ─────────────────────────────────────────────
# 10. Model
# ─────────────────────────────────────────────
class GNNMaxViTFusion(nn.Module):
    def __init__(self, meta_input_dim, num_classes):
        super().__init__()

        self.backbone = timm.create_model(
            "maxvit_tiny_tf_224",
            pretrained=True, num_classes=0, global_pool=""
        )
        for i, stage in enumerate(self.backbone.stages):
            if i < CFG.FREEZE_BLOCKS:
                for param in stage.parameters():
                    param.requires_grad = False

        with torch.no_grad():
            dummy = torch.zeros(1, 3, CFG.IMG_SIZE, CFG.IMG_SIZE)
            feat  = self.backbone(dummy)
        C      = feat.shape[1]
        self.R = feat.shape[2] * feat.shape[3]

        self.img_proj = nn.Sequential(
            nn.Linear(C, CFG.D), nn.LayerNorm(CFG.D), nn.GELU())

        self.meta_proj = nn.Sequential(
            nn.Linear(meta_input_dim, CFG.D * CFG.META_TOKENS),
            nn.LayerNorm(CFG.D * CFG.META_TOKENS), nn.GELU())

        self.gat1  = GATv2Conv(CFG.D, CFG.D // CFG.GAT_HEADS,
                               heads=CFG.GAT_HEADS, concat=True,
                               dropout=CFG.DROPOUT, add_self_loops=True)
        self.norm1 = nn.LayerNorm(CFG.D)

        self.gat2  = GATv2Conv(CFG.D, CFG.D // CFG.GAT_HEADS,
                               heads=CFG.GAT_HEADS, concat=True,
                               dropout=CFG.DROPOUT, add_self_loops=True)
        self.norm2 = nn.LayerNorm(CFG.D)

        self.head = nn.Sequential(
            nn.Linear(CFG.D, CFG.D // 2), nn.GELU(),
            nn.Dropout(CFG.DROPOUT),
            nn.Linear(CFG.D // 2, num_classes))

    def forward(self, images, metadata):
        B    = images.size(0)
        feat = self.backbone(images)
        feat = feat.flatten(2).permute(0, 2, 1)
        feat = self.img_proj(feat)

        meta = self.meta_proj(metadata)
        meta = meta.view(B, CFG.META_TOKENS, -1)

        graphs      = [build_graph(feat[b], meta[b]) for b in range(B)]
        batch_graph = Batch.from_data_list(graphs).to(images.device)

        x = batch_graph.x
        e = batch_graph.edge_index
        x = self.norm1(F.gelu(self.gat1(x, e)))
        x = self.norm2(F.gelu(self.gat2(x, e)))

        N      = self.R + CFG.META_TOKENS
        x      = x.view(B, N, -1)
        pooled = x[:, :self.R, :].mean(dim=1)
        return self.head(pooled)

# ─────────────────────────────────────────────
# 11. TTA
# ─────────────────────────────────────────────
def predict_with_tta(model, image_tensor, metadata_tensor):
    variants = [
        image_tensor,
        TF.hflip(image_tensor),
        TF.vflip(image_tensor),
        TF.rotate(image_tensor,  90),
        TF.rotate(image_tensor, 270),
    ]
    model.eval()
    probs_list = []
    with torch.no_grad():
        for v in variants:
            out  = model(v.to(device), metadata_tensor.to(device))
            prob = torch.softmax(out, dim=1).cpu()
            probs_list.append(prob)
    return torch.stack(probs_list).mean(dim=0)

# ─────────────────────────────────────────────
# 12. Train / Validate
# ─────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, scheduler):
    model.train()
    total_loss, correct, total = 0, 0, 0
    optimizer.zero_grad()

    for step, (imgs, metas, labels) in enumerate(
            tqdm(loader, desc="Train", leave=False)):

        imgs, metas, labels = (imgs.to(device),
                               metas.to(device),
                               labels.to(device))
        out  = model(imgs, metas)
        loss = criterion(out, labels) / CFG.GRAD_ACCUM
        loss.backward()

        if (step + 1) % CFG.GRAD_ACCUM == 0 or (step + 1) == len(loader):
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * CFG.GRAD_ACCUM
        correct    += (out.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(loader), correct / total


def valid_one_epoch(model, loader, use_tta=False):
    model.eval()
    preds, trues, probs_all = [], [], []

    with torch.no_grad():
        for imgs, metas, labels in tqdm(loader, desc="Val  ", leave=False):
            if use_tta:
                batch_probs = []
                for i in range(imgs.size(0)):
                    p = predict_with_tta(
                        model,
                        imgs[i].unsqueeze(0),
                        metas[i].unsqueeze(0)
                    )
                    batch_probs.append(p)
                prob = torch.cat(batch_probs, dim=0)
            else:
                imgs, metas = imgs.to(device), metas.to(device)
                out  = model(imgs, metas)
                prob = torch.softmax(out, dim=1).cpu()

            preds.extend(prob.argmax(1).numpy())
            trues.extend(labels.numpy())
            probs_all.extend(prob.numpy())

    trues     = np.array(trues)
    preds     = np.array(preds)
    probs_all = np.array(probs_all)

    acc = accuracy_score(trues, preds)
    f1  = f1_score(trues, preds, average="weighted")
    auc = (roc_auc_score(trues, probs_all[:, 1]) if NUM_CLASSES == 2
           else roc_auc_score(trues, probs_all,
                              multi_class="ovr", average="weighted"))

    return acc, f1, auc, trues, preds, probs_all

# ─────────────────────────────────────────────
# 13. Training ✅ fixed train/val split
# ─────────────────────────────────────────────
train_loader = DataLoader(
    GNNDataset(train_df, get_train_transform()),
    batch_size  = CFG.BATCH_SIZE,
    shuffle     = True,
    num_workers = CFG.NUM_WORKERS,
    pin_memory  = True
)
val_loader = DataLoader(
    GNNDataset(val_df, get_val_transform()),
    batch_size  = CFG.BATCH_SIZE,
    shuffle     = False,
    num_workers = CFG.NUM_WORKERS,
    pin_memory  = True
)

model = GNNMaxViTFusion(META_DIM, NUM_CLASSES).to(device)

trainable = sum(p.numel() for p in model.parameters()
                if p.requires_grad) / 1e6
total_p   = sum(p.numel() for p in model.parameters()) / 1e6
print(f"\nTotal params    : {total_p:.1f}M")
print(f"Trainable params: {trainable:.1f}M  "
      f"(first {CFG.FREEZE_BLOCKS} stages frozen)")

optimizer = torch.optim.AdamW([
    {"params": [p for p in model.backbone.parameters()
                if p.requires_grad],
     "lr": CFG.LR,      "weight_decay": CFG.WEIGHT_DECAY},
    {"params": model.img_proj.parameters(),
     "lr": CFG.HEAD_LR, "weight_decay": CFG.WEIGHT_DECAY},
    {"params": model.meta_proj.parameters(),
     "lr": CFG.HEAD_LR, "weight_decay": CFG.WEIGHT_DECAY},
    {"params": model.gat1.parameters(),
     "lr": CFG.HEAD_LR, "weight_decay": CFG.WEIGHT_DECAY},
    {"params": model.gat2.parameters(),
     "lr": CFG.HEAD_LR, "weight_decay": CFG.WEIGHT_DECAY},
    {"params": model.head.parameters(),
     "lr": CFG.HEAD_LR, "weight_decay": CFG.WEIGHT_DECAY},
])

total_steps = CFG.EPOCHS * len(train_loader)
scheduler   = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr      = [CFG.LR, CFG.HEAD_LR, CFG.HEAD_LR,
                   CFG.HEAD_LR, CFG.HEAD_LR, CFG.HEAD_LR],
    total_steps = total_steps,
    pct_start   = 0.1,
    anneal_strategy = "cos"
)

criterion    = FocalLoss()
best_f1      = 0.0
patience_cnt = 0

print(f"\n{'='*55}")
print("  GNN + MaxViT-T  —  Training")
print(f"{'='*55}")

for epoch in range(CFG.EPOCHS):
    tr_loss, tr_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, scheduler)
    acc, f1, auc, _, _, _ = valid_one_epoch(
        model, val_loader, use_tta=False)

    print(f"Epoch {epoch+1:02d}/{CFG.EPOCHS} | "
          f"Loss: {tr_loss:.4f} | Train Acc: {tr_acc:.4f} | "
          f"Val Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")

    if f1 > best_f1:
        best_f1      = f1
        patience_cnt = 0
        torch.save(model.state_dict(), "best_gnn_maxvit.pth")
        print(f"  Saved (F1: {best_f1:.4f})")
    else:
        patience_cnt += 1
        if patience_cnt >= CFG.PATIENCE:
            print(f"  ⏹ Early stopping at epoch {epoch+1}")
            break

print(f"\nBest Val F1: {best_f1:.4f}")

# ─────────────────────────────────────────────
# 14. Final Evaluation on Test Set ✅
# ─────────────────────────────────────────────
print(f"\n{'='*55}")
print("  Final Evaluation on TEST SET")
print(f"{'='*55}")

model.load_state_dict(
    torch.load("best_gnn_maxvit.pth", map_location=device))

test_loader = DataLoader(
    GNNDataset(test_df, get_val_transform()),
    batch_size  = CFG.BATCH_SIZE,
    shuffle     = False,
    num_workers = CFG.NUM_WORKERS,
    pin_memory  = True
)

# Without TTA
acc, f1, auc, trues, preds, _ = valid_one_epoch(
    model, test_loader, use_tta=False)
print(f"\n[Without TTA] Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")

# With TTA
acc_t, f1_t, auc_t, trues, preds_tta, _ = valid_one_epoch(
    model, test_loader, use_tta=True)
print(f"[With    TTA] Acc: {acc_t:.4f} | F1: {f1_t:.4f} | AUC: {auc_t:.4f}")

print("\n" + "="*55)
print("FINAL RESULTS — GNN + MaxViT-T  (with TTA)")
print("="*55)
print(classification_report(trues, preds_tta, target_names=classes_str))

# ✅ Save for visualization notebook
np.save("gnn_trues.npy",     trues)
np.save("gnn_preds_tta.npy", preds_tta)
print("\n[✓] Saved gnn_trues.npy & gnn_preds_tta.npy for visualization")

Using device: cuda
Classes     : ['0', '1']
Num classes : 2
Train size  : 11941
Val size    : 2559
Test size   : 2559

Train distribution:
label
1    5971
0    5970
Name: count, dtype: int64

Meta columns (29): ['age_scaled', 'melanocytic', 'sex_Unknown', 'sex_female', 'sex_male', 'anatom_site_general_Unknown', 'anatom_site_general_anterior torso', 'anatom_site_general_head/neck', 'anatom_site_general_lateral torso', 'anatom_site_general_lower extremity', 'anatom_site_general_oral/genital', 'anatom_site_general_palms/soles', 'anatom_site_general_posterior torso', 'anatom_site_general_upper extremity', 'dermoscopic_type_Unknown', 'dermoscopic_type_contact non-polarized', 'dermoscopic_type_contact polarized', 'dermoscopic_type_non-contact polarized', 'diagnosis_confirm_type_Unknown', 'diagnosis_confirm_type_confocal microscopy with consensus dermoscopy', 'diagnosis_confirm_type_histopathology', 'diagnosis_confirm_type_serial imaging showing no change', 'diagnosis_confirm_type_single imag

model.safetensors:   0%|          | 0.00/124M [00:00<?, ?B/s]


Total params    : 30.9M
Trainable params: 29.5M  (first 2 stages frozen)

  GNN + MaxViT-T  —  Training


Epoch 01/20 | Loss: 0.0423 | Train Acc: 0.6097 | Val Acc: 0.6932 | F1: 0.6931 | AUC: 0.7735
  Saved (F1: 0.6931)


Epoch 02/20 | Loss: 0.0356 | Train Acc: 0.7388 | Val Acc: 0.7905 | F1: 0.7879 | AUC: 0.9045
  Saved (F1: 0.7879)


Epoch 03/20 | Loss: 0.0290 | Train Acc: 0.8168 | Val Acc: 0.8386 | F1: 0.8384 | AUC: 0.9265
  Saved (F1: 0.8384)


KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 3 — Visualizations
# ============================================================

# 1 — Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples",
            xticklabels=[str(c) for c in classes],
            yticklabels=[str(c) for c in classes], ax=ax)
ax.set_ylabel("True"); ax.set_xlabel("Predicted")
ax.set_title(f"Confusion Matrix — Fold {best_fold+1}")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f"confusion_matrix_fold{best_fold+1}.png"), dpi=150)
plt.show()

# 2 — ROC Curve
fpr, tpr, thresholds = roc_curve(trues, probs)
youden_idx = np.argmax(tpr - fpr)
opt_thr    = thresholds[youden_idx]

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, lw=2, color="purple", label=f"AUC = {auc:.4f}")
ax.scatter(fpr[youden_idx], tpr[youden_idx], color="red", zorder=5,
           label=f"Optimal thr = {opt_thr:.3f}")
ax.plot([0, 1], [0, 1], "k--")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title(f"ROC Curve — Fold {best_fold+1}")
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f"roc_curve_fold{best_fold+1}.png"), dpi=150)
plt.show()

# 3 — Training Curves
history = all_histories[f"fold_{best_fold+1}"]
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

axes[0].plot(history["train_loss"], label="Train", marker="o", ms=3)
axes[0].plot(history["val_loss"],   label="Val",   marker="o", ms=3)
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True)

axes[1].plot(history["train_auc"],   label="Train",   alpha=0.6)
axes[1].plot(history["val_auc"],     label="Val",     alpha=0.6)
axes[1].plot(history["val_auc_ema"], label="Val EMA", lw=2)
axes[1].set_title("AUC-ROC"); axes[1].legend(); axes[1].grid(True)

axes[2].plot(history["val_f1"], label="Val F1", color="green", marker="o", ms=3)
axes[2].set_title("Val F1"); axes[2].legend(); axes[2].grid(True)

axes[3].plot(history["threshold"], label="Optimal Threshold", color="purple", marker="o", ms=3)
axes[3].axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="Default 0.5")
axes[3].set_title("Optimal Threshold"); axes[3].legend(); axes[3].grid(True)

plt.suptitle(f"ViT Early Fusion — Training Curves (Fold {best_fold+1})",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f"training_curves_fold{best_fold+1}.png"), dpi=150)
plt.show()